# Tam Eğitim Notebook'u (A100) — Türkçe OCR İnce Ayarı

Bu notebook, **A100 GPU** ile tam veri ve tam epoch sayısıyla asıl ince ayarı çalıştırır.
Buraya geçmeden ÖNCE `00_pilot_t4_setup_eda.ipynb`'nin T4'te HATASIZ tamamlandığından
emin olun — pilot, bu notebook'un güvenle çalışacağının kanıtıdır.

Çalıştırmadan önce Colab menüsünden: **Çalışma zamanı > Çalışma zamanı türünü değiştir >
A100 GPU** seçili olmalıdır (Colab Pro/Pro+ gerektirir).

Hücreleri SIRAYLA çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/nidazeren/qwen2.5-vl.git"
REPO_DIR = "/content/qwen2.5-vl"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

# hf_xet (HuggingFace'in hizlandirilmis "Xet" indirme sistemi) bu ortamda tekrar tekrar
# indirmeleri %85-99 civarinda sessizce tikanmaya sokuyor (dosya/agirlik indirmeleri
# saatlerce "Reconstructing (incomplete total...)" durumunda kaliyor). Standart, daha
# yavas ama GUVENILIR HTTP indirmesine dusmesi icin kaldiriyoruz.
!pip uninstall -y hf_xet -q

# torchao: Colab'in onceden kurdugu surum, peft'in bekledigi surumle (>0.16.0)
# UYUMSUZ. peft, LoRA hedef modullerini eslerken (embed_tokens dahil) her modul icin
# "bu torchao ile nicemlenmis mi?" kontrolu yapiyor; bu kontrolun KENDISI eski
# surumde ImportError firlatiyor (train_sft.py -> apply_lora() adiminda cokme).
# Projemiz torchao KULLANMIYOR (4-bit icin bitsandbytes kullaniliyor), bu yuzden en
# temiz cozum paketi tamamen kaldirmak.
!pip uninstall -y torchao -q

## (İsteğe bağlı) flash-attention-2 kurulumu
A100'de flash-attention-2 kuruluysa `training/lora_setup.py` otomatik olarak onu
seçer (bkz. `select_dtype_and_attn_impl`); kurulu değilse otomatik olarak `sdpa`'ya
(daha yavaş ama tamamen doğru/güvenli) düşer. Derlemesi birkaç dakika sürebilir; hız
kazancı istemiyorsanız bu hücreyi atlayabilirsiniz.

In [ ]:
!pip install -q flash-attn --no-build-isolation

## Ortam değişkenleri: TAM MOD (PILOT_MODE=0)
Bu, `configs/config.py` içindeki tüm boyutları (veri, epoch, batch) tam-ölçekli
değerlere geçirir VE çıktı klasörlerini (`processed_data/full`, `checkpoints/full`,
`eval_outputs/full`) pilot çıktılarından AYRI tutar (bkz. config.py: MODE_TAG).

In [ ]:
import os, sys

os.environ["QWEN_OCR_PILOT_MODE"] = "0"
os.environ["QWEN_OCR_DRIVE_ROOT"] = "/content/drive/MyDrive/qwen25vl_turkish_ocr"
sys.path.insert(0, REPO_DIR)

from configs import config
config.ensure_directories()
print("PILOT_MODE:", config.PILOT_MODE)
print("MODE_TAG:", config.MODE_TAG)
print("Hedef kova boyutlari (tam):", config.compute_bucket_target_sizes())

## Ortam değişkenleri (devam): (İsteğe bağlı) RUN_NAME öncesi eski çıktıları arşivle
`configs/config.py`'ye eklenen `RUN_NAME` alt-klasörleme yapısı checkpoint/log/eval
çıktılarının konumunu değiştirdi. Daha önce bu notebook'u çalıştırdıysanız, eski
(RUN_NAME öncesi) çıktılar hâlâ Drive'da duruyor olabilir. Bu hücre onları **silmez**,
Drive'da bir `_archive_pre_run_name_*` klasörüne taşır.

In [ ]:
import shutil
from datetime import datetime

# RUN_NAME'e göre alt-klasörleme (bkz. configs/config.py) getirilmeden ÖNCE üretilmiş
# çıktılar farklı bir dizin yapısındaydı: checkpoints/{mode}/trainer_output (RUN_NAME
# alt klasörü OLMADAN), logs/{mode}/<event dosyaları doğrudan burada>,
# eval_outputs/{mode}/epoch_N.json + regression_report.json (runs/{RUN_NAME}/ OLMADAN).
# Bu hücre bunları SİLMEZ, geri alınabilir şekilde bir arşiv klasörüne TAŞIR --
# baseline.json ve onun checkpoints/baseline_*.jsonl önbelleği KORUNUR (hâlâ paylaşılan/
# geçerlidir, RUN_NAME'den etkilenmez, bkz. configs/config.py notu).
ARCHIVE_DIR = config.DRIVE_ROOT / f"_archive_pre_run_name_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OLD_CHECKPOINT_SUBDIRS = ["trainer_output", "final_adapter", "best_pareto_adapter"]

moved = []
for mode_tag in ["pilot", "full"]:
    ckpt_dir = config.DRIVE_ROOT / "checkpoints" / mode_tag
    if ckpt_dir.exists():
        for name in OLD_CHECKPOINT_SUBDIRS:
            old_path = ckpt_dir / name
            if old_path.exists() and old_path.is_dir():
                dest = ARCHIVE_DIR / "checkpoints" / mode_tag / name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(old_path), str(dest))
                moved.append(str(old_path))

    log_dir = config.DRIVE_ROOT / "logs" / mode_tag
    if log_dir.exists():
        for item in log_dir.iterdir():
            if item.is_file():  # RUN_NAME öncesi: event dosyaları doğrudan burada
                dest = ARCHIVE_DIR / "logs" / mode_tag / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))

    eval_dir = config.DRIVE_ROOT / "eval_outputs" / mode_tag
    if eval_dir.exists():
        for pattern in ["epoch_*.json", "regression_report.json"]:
            for item in eval_dir.glob(pattern):
                dest = ARCHIVE_DIR / "eval_outputs" / mode_tag / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))
        ckpt_cache = eval_dir / "checkpoints"
        if ckpt_cache.exists():
            for item in ckpt_cache.glob("epoch_*.jsonl"):
                dest = ARCHIVE_DIR / "eval_outputs" / mode_tag / "checkpoints" / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))

if moved:
    print(f"{len(moved)} eski (RUN_NAME öncesi) öğe arşivlendi -> {ARCHIVE_DIR}")
    for m in moved:
        print(" ", m)
else:
    print("Taşınacak eski (RUN_NAME öncesi) çıktı bulunamadı -- temiz.")

## (İsteğe bağlı) Kaggle kimlik bilgisi — TS-TR için
`USE_SCENE_TEXT=False` (varsayılan) iken bu hücreyi ATLAYABİLİRSİNİZ. `USE_SCENE_TEXT=True`
yaptıysanız ve pilot notebook'ta bir kez ayarladıysanız burada otomatik bulunur.

In [ ]:
import os, json, shutil, stat

if not config.USE_SCENE_TEXT:
    print("USE_SCENE_TEXT=False; Kaggle kimlik bilgisi gerekmiyor, hucre atlaniyor.")
else:
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    target = os.path.join(kaggle_dir, "kaggle.json")
    drive_copy = str(config.DRIVE_ROOT / "kaggle.json")

    def _try_colab_secrets() -> bool:
        try:
            from google.colab import userdata

            username = userdata.get("KAGGLE_USERNAME")
            key = userdata.get("KAGGLE_KEY")
        except Exception:
            return False
        if not username or not key:
            return False
        with open(target, "w", encoding="utf-8") as f:
            json.dump({"username": username, "key": key}, f)
        return True

    if os.path.exists(drive_copy):
        shutil.copy(drive_copy, target)
    elif os.path.exists(target):
        pass
    elif _try_colab_secrets():
        pass
    else:
        from google.colab import files
        print("Lutfen kaggle.json dosyanizi secin:")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        shutil.move(uploaded_name, target)

    shutil.copy(target, drive_copy)
    os.chmod(target, stat.S_IRUSR | stat.S_IWUSR)
    print("Kaggle kimlik bilgisi hazir:", target)

## Tam veri hazırlama
`PILOT_MODE=False` olduğundan `data/prepare_datasets.py`, `RAW_DATA_DIR` altındaki
(pilotta küçük boyutlu kaydedilmiş) kaynakları TAM boyutlarıyla YENİDEN indirir/kaydeder.

In [ ]:
!python data/prepare_datasets.py

## (İsteğe bağlı) EDA'yı tam veriyle tekrar gözden geçirin

In [ ]:
!python analysis/tokenizer_analysis.py
!python analysis/vision_token_eda.py

## Self-distillation replay verisi (tam boyut)
`data/replay_generation.py`, mevcut replay verisinin YENİ (tam) hedefleri karşılayıp
karşılamadığını kontrol eder; karşılamıyorsa (pilotun küçük verisi yetersiz kalacağından)
otomatik olarak yeniden üretir. Bu adım A100'de bile en uzun süren adımlardan biri olabilir.

In [ ]:
!python data/replay_generation.py

In [ ]:
!python data/build_chat_dataset.py

## Baseline değerlendirme (tam Test A/B setleri üzerinde)
MODE_TAG=full olduğundan bu, pilotun baseline.json'ından TAMAMEN AYRI, tam Test A/B
setlerine göre yeni bir baseline üretir/kullanır.

In [ ]:
!python evaluation/evaluate.py --tag baseline

## Ablation Metodolojisi — Türkçe OCR ince ayarında catastrophic forgetting'i azaltma

Aşağıdaki hücreler, tek bir `!python training/train_sft.py` çağrısı yerine, proje notlarında
tanımlanan **faz-faz ablation planını** çalıştırır. Veri karışımı (`MIXTURE_PRINTED/HANDWRITING/REPLAY`)
faz 1-4 boyunca **DEĞİŞMEZ**; yalnızca `configs/config.py`'deki ablation bayrakları
(`QWEN_OCR_*` ortam değişkenleriyle, dosya elle düzenlenmeden) koşumdan koşuma değişir.
Her koşum kendi `RUN_NAME` altında izole checkpoint/log/eval çıktısı üretir (bkz.
`configs/config.py`: `EVAL_OUTPUT_DIR` vs `EVAL_RUN_OUTPUT_DIR`), böylece hiçbir koşum
bir öncekinin sonucunu EZMEZ ve sonda hepsi karşılaştırılabilir.

Sıra (her fazın çıktısı bir SONRAKİ fazın sabit tuttuğu değeri belirler):
1. **Learning rate** (1e-4, 5e-5, 1e-5, 5e-6, 1e-6) — LoRA 36/36 katman, attn+mlp sabit.
2. **LoRA layer coverage** (`all`, `last18`, `last12`) — en iyi LR sabit.
3. **Attention-only vs attention+MLP** (+ özellikle vurgulanan `last12 + attn_only` deneyi).
4. **LoRA rank/alpha/dropout** — rank ve dropout ayrı küçük taramalarla (tam çapraz grid
   yerine, koşum sayısını makul tutmak için); alpha, rank ile sabit oranda (`alpha=2*r`) değişir.
5. **(Yalnızca gerekirse) replay loss weighting** — `MIXTURE_REPLAY` veri oranını DEĞİL,
   `LOSS_WEIGHT_REPLAY` ağırlığını artırır (bkz. `training/distillation_trainer.py`).

Her fazdan sonra `analysis/ablation_report.py` bir karşılaştırma tablosu basar (CER/WER/
exact-match/Türkçe karakter doğruluğu/Test A forgetting + Pareto-uygun checkpoint var mı).
**Bir sonraki fazı başlatmadan önce bu tabloyu inceleyip `BEST_*` değişkenini elle
güncelleyin** — hangi hiperparametrenin taşınacağı, otomatikleştirilmeyecek kadar önemli
bir karardır (bkz. `training/callbacks.py` docstring'i, aynı felsefe).

Her koşum, `config.TEST_A_REGRESSION_RELATIVE_THRESHOLD` (%15) içinde kalan epoch'lar
arasından Test B CER'i en düşük olanı otomatik olarak
`checkpoints/full/{RUN_NAME}/best_pareto_adapter`'a kaydeder (bkz. `training/callbacks.py`);
hiçbir epoch koşulu sağlamazsa `pareto_summary.json` bunu açıkça belirtir.

In [ ]:
import os, sys, json, subprocess

def run_training(run_name: str, env_overrides: dict) -> dict:
    """Verilen RUN_NAME ve ortam değişkeni override'larıyla training/train_sft.py'yi AYRI
    bir alt-process olarak çalıştırır (VRAM'in her koşum arasında TAMAMEN temizlenmesi
    için -- aynı process içinde art arda ~3B'lik modelleri yükleyip silmek CUDA belleğinde
    parçalanmaya yol açabilir). Colab hücre çıktısında canlı akar. Koşum bitince
    eval_history.json'ı okuyup döner (yoksa boş sözlük -- koşum hata verdiyse)."""
    env = os.environ.copy()
    env["QWEN_OCR_RUN_NAME"] = run_name
    for key, value in env_overrides.items():
        env[key] = str(value)

    print(f"\n{'='*80}\n[ablation] run_name={run_name!r} overrides={env_overrides}\n{'='*80}")
    result = subprocess.run([sys.executable, "training/train_sft.py"], env=env, cwd=REPO_DIR)
    if result.returncode != 0:
        print(f"[ablation] !! run_name={run_name!r} HATA ile sonlandı (returncode={result.returncode}).")

    history_path = config.EVAL_OUTPUT_DIR / "runs" / run_name / "eval_history.json"
    return json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else {}

In [ ]:
LR_CANDIDATES = ["1e-4", "5e-5", "1e-5", "5e-6", "1e-6"]

phase1_results = {}
for lr in LR_CANDIDATES:
    run_name = f"p1_lr_{lr}"
    phase1_results[run_name] = run_training(run_name, {"QWEN_OCR_LEARNING_RATE": lr})

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

In [ ]:
BEST_LR = "5e-5"  # <-- yukarıdaki karşılaştırma tablosunu inceleyip güncelleyin

In [ ]:
LAYER_SCOPE_CANDIDATES = ["all", "last18", "last12"]

phase2_results = {}
for scope in LAYER_SCOPE_CANDIDATES:
    run_name = f"p2_layers_{scope}"
    phase2_results[run_name] = run_training(run_name, {
        "QWEN_OCR_LEARNING_RATE": BEST_LR,
        "QWEN_OCR_LORA_LAYER_SCOPE": scope,
    })

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

In [ ]:
BEST_LAYER_SCOPE = "last18"  # <-- yukarıdaki karşılaştırma tablosunu inceleyip güncelleyin

In [ ]:
TARGET_SCOPE_CANDIDATES = ["attn_mlp", "attn_only"]

phase3_results = {}
for target_scope in TARGET_SCOPE_CANDIDATES:
    run_name = f"p3_{BEST_LAYER_SCOPE}_{target_scope}"
    phase3_results[run_name] = run_training(run_name, {
        "QWEN_OCR_LEARNING_RATE": BEST_LR,
        "QWEN_OCR_LORA_LAYER_SCOPE": BEST_LAYER_SCOPE,
        "QWEN_OCR_LORA_TARGET_SCOPE": target_scope,
    })

# Proje notlarında özellikle vurgulanan deney: son 12 katman + yalnızca attention --
# BEST_LAYER_SCOPE farklı çıksa bile (kendi başına önemli bir karşılaştırma noktası
# olduğu için) AYRICA çalıştırılır.
if BEST_LAYER_SCOPE != "last12":
    phase3_results["p3_last12_attn_only_highlight"] = run_training(
        "p3_last12_attn_only_highlight",
        {
            "QWEN_OCR_LEARNING_RATE": BEST_LR,
            "QWEN_OCR_LORA_LAYER_SCOPE": "last12",
            "QWEN_OCR_LORA_TARGET_SCOPE": "attn_only",
        },
    )

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

In [ ]:
BEST_TARGET_SCOPE = "attn_only"  # <-- yukarıdaki karşılaştırma tablosunu inceleyip güncelleyin

In [ ]:
# alpha, rank ile SABİT ORANDA (alpha=2*r, mevcut varsayılan 16/32 ile aynı oran) değişir;
# "daha fazla LoRA = daha iyi OCR" varsayımı yerine OCR kazancı ile forgetting arasındaki
# Pareto dengesi aranıyor (bkz. proje notları).
RANK_CANDIDATES = [8, 16, 32]

phase4_rank_results = {}
for r in RANK_CANDIDATES:
    run_name = f"p4_r{r}"
    phase4_rank_results[run_name] = run_training(run_name, {
        "QWEN_OCR_LEARNING_RATE": BEST_LR,
        "QWEN_OCR_LORA_LAYER_SCOPE": BEST_LAYER_SCOPE,
        "QWEN_OCR_LORA_TARGET_SCOPE": BEST_TARGET_SCOPE,
        "QWEN_OCR_LORA_R": r,
        "QWEN_OCR_LORA_ALPHA": 2 * r,
    })

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

In [ ]:
BEST_R = 16  # <-- yukarıdaki karşılaştırma tablosunu inceleyip güncelleyin

In [ ]:
DROPOUT_CANDIDATES = [0.0, 0.05, 0.1]

phase4_dropout_results = {}
for dropout in DROPOUT_CANDIDATES:
    run_name = f"p4_dropout{dropout}"
    phase4_dropout_results[run_name] = run_training(run_name, {
        "QWEN_OCR_LEARNING_RATE": BEST_LR,
        "QWEN_OCR_LORA_LAYER_SCOPE": BEST_LAYER_SCOPE,
        "QWEN_OCR_LORA_TARGET_SCOPE": BEST_TARGET_SCOPE,
        "QWEN_OCR_LORA_R": BEST_R,
        "QWEN_OCR_LORA_ALPHA": 2 * BEST_R,
        "QWEN_OCR_LORA_DROPOUT": dropout,
    })

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

In [ ]:
BEST_DROPOUT = 0.05  # <-- yukarıdaki karşılaştırma tablosunu inceleyip güncelleyin

## Faz 5 (yalnızca gerekirse) — Replay Loss Weighting

Faz 1-4 sonunda seçilen en iyi konfigürasyonun Pareto özetini (`pareto_summary.json`)
kontrol edin. Test A forgetting **hâlâ %15 eşiğine yakın veya üzerindeyse**, `MIXTURE_REPLAY`
veri oranını DEĞİŞTİRMEDEN önce aşağıdaki hücreyle yalnızca replay örneklerinin kayıp
ağırlığını artırmayı deneyin (`ENABLE_WEIGHTED_LOSS=1` + `LOSS_WEIGHT_REPLAY`). Forgetting
zaten eşiğin altındaysa bu fazı ATLAYIP doğrudan "Nihai model seçimi" bölümüne geçebilirsiniz.

In [ ]:
REPLAY_WEIGHT_CANDIDATES = [1.0, 1.5, 2.0]

phase5_results = {}
for w in REPLAY_WEIGHT_CANDIDATES:
    run_name = f"p5_replay_w{w}"
    phase5_results[run_name] = run_training(run_name, {
        "QWEN_OCR_LEARNING_RATE": BEST_LR,
        "QWEN_OCR_LORA_LAYER_SCOPE": BEST_LAYER_SCOPE,
        "QWEN_OCR_LORA_TARGET_SCOPE": BEST_TARGET_SCOPE,
        "QWEN_OCR_LORA_R": BEST_R,
        "QWEN_OCR_LORA_ALPHA": 2 * BEST_R,
        "QWEN_OCR_LORA_DROPOUT": BEST_DROPOUT,
        "QWEN_OCR_ENABLE_WEIGHTED_LOSS": "1",
        "QWEN_OCR_LOSS_WEIGHT_REPLAY": w,
    })

!python analysis/ablation_report.py --mode-tag {config.MODE_TAG}

## Nihai model seçimi ve drift analizi

Aşağıdaki hücre, o ana kadar çalıştırılan TÜM koşumlar arasından Pareto-uygun (Test A
forgetting ≤ %15) checkpoint'ler içinde Test B CER'i en düşük olanı bulur -- bu, projenin
"OCR kazancı ile forgetting arasındaki Pareto dengesi" hedefine göre SEÇİLMESİ gereken
nihai modeldir (yalnızca en yüksek OCR skoruna sahip model DEĞİL). Ardından o koşumun
katman başına LoRA drift grafiğini çizer (representation drift'in hangi katmanlarda
yoğunlaştığını ve Test A regresyonunun başladığı adımla örtüşüp örtüşmediğini görmek için).

In [ ]:
import json

runs_root = config.EVAL_OUTPUT_DIR / "runs"
best_run_name, best_summary = None, None
for run_dir in (sorted(runs_root.iterdir()) if runs_root.exists() else []):
    pareto_path = run_dir / "pareto_summary.json"
    if not pareto_path.exists():
        continue
    pareto = json.loads(pareto_path.read_text(encoding="utf-8"))
    if pareto.get("best_epoch") is None:
        continue
    cer = pareto["best_epoch"]["test_b_cer"]
    if best_summary is None or cer < best_summary["best_epoch"]["test_b_cer"]:
        best_run_name, best_summary = run_dir.name, pareto

if best_run_name is None:
    print(
        "!! Hiçbir koşum Pareto koşulunu (Test A forgetting <= eşik) sağlamadı. "
        "Faz 5'i (replay loss weighting) çalıştırmayı veya eşiği "
        "(TEST_A_REGRESSION_RELATIVE_THRESHOLD) yeniden değerlendirmeyi düşünün."
    )
else:
    print(f"Nihai seçilen konfigürasyon: run_name={best_run_name!r}")
    print(json.dumps(best_summary["best_epoch"], ensure_ascii=False, indent=2))
    print(f"Adapter yolu: {best_summary['best_adapter_path']}")
    print(
        "\nDeğerlendirmek için:\n"
        f"  python evaluation/evaluate.py --tag final_check --adapter-path {best_summary['best_adapter_path']}"
    )
    print(
        "\nDrift grafiği için:\n"
        f"  python analysis/ablation_report.py --mode-tag {config.MODE_TAG} --drift-plot {best_run_name}"
    )

    from analysis.ablation_report import plot_drift
    plot_drift(config.MODE_TAG, best_run_name)

## Tek bir koşumun ham çıktısını incelemek isterseniz
Aşağıdaki hücrede `RUN_NAME_TO_INSPECT`'i (ör. `"p2_layers_last12"`) istediğiniz bir
koşumla değiştirip çalıştırın; o koşumun tüm epoch JSON çıktılarını basar.

In [ ]:
RUN_NAME_TO_INSPECT = "p1_lr_5e-5"  # <-- incelemek istediğiniz RUN_NAME ile değiştirin

run_dir = config.EVAL_OUTPUT_DIR / "runs" / RUN_NAME_TO_INSPECT
for path in sorted(run_dir.glob("*.json")):
    print(f"--- {path.name} ---")
    print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    print()

## Bir koşum regresyon nedeniyle durduysa
1) `eval_outputs/full/runs/{RUN_NAME}/regression_report.json` içindeki önerilen sonraki
   adımları okuyun (artık `configs/config.py`'yi elle DEĞİL, `QWEN_OCR_*` ortam
   değişkenlerini değiştirmeyi önerir).
2) Yukarıdaki `run_training(run_name, env_overrides)` yardımcı fonksiyonunu, güncellenmiş
   `env_overrides` ile ve YENİ bir `run_name` ile tekrar çağırın (aynı `run_name`'i
   tekrar kullanmak, o koşumun önceki checkpoint/eval çıktılarının üzerine yazar).
3) `SFTConfig(save_strategy="epoch")` sayesinde `checkpoints/full/{RUN_NAME}/trainer_output`
   altında ara checkpoint'ler mevcuttur; isterseniz `training/train_sft.py` içindeki
   `trainer.train()` çağrısını `trainer.train(resume_from_checkpoint=True)` olacak şekilde
   uyarlayabilirsiniz.